# 01 - Data Quality

## Objective
Profile the analytical datasets used by the validation layer and verify their expected grains before interpreting results.

## Data source
PostgreSQL analytics views and the daily/hourly fact tables in the `parkitup` schema.

## Methodology
Use the reusable database loader and profiling functions to report shape, types, missingness, duplicates, unique counts, percentiles, central tendency, dispersion, skew, zero inflation, and source-grain checks.

In [1]:
from pathlib import Path
import sys
from IPython.display import display

cwd = Path.cwd().resolve()
REPO_ROOT = next(path for path in (cwd, *cwd.parents) if (path / 'database').exists() and (path / 'python').exists())
sys.path.insert(0, str(REPO_ROOT))

from python.analysis.data_access import load_analysis_inputs, source_contract_checks
from python.analysis.profiling import profile_datasets, suspicious_distributions

In [2]:
dataset_names = [
    'parking', 'owners', 'competition', 'location_demand', 'acquisition_terms',
    'daily_performance', 'hourly_profile', 'outreach', 'acquisition_scores',
    'component_scores', 'parking_performance', 'locality_summary',
]
inputs = load_analysis_inputs(dataset_names)
contracts = source_contract_checks(inputs)
profile, summary = profile_datasets(inputs)
suspicious = suspicious_distributions(profile)

display(contracts)
display(summary)

,check_id,description,observed,status
0,SRC-parking_performance,parking_performance has its documented row grain,120,PASS
1,SRC-acquisition_scores,acquisition_scores has its documented row grain,120,PASS
2,SRC-component_scores,component_scores has its documented row grain,120,PASS
3,SRC-locality_summary,locality_summary has its documented row grain,17,PASS
4,SRC-daily_performance,daily_performance has its documented row grain,43800,PASS
5,SRC-hourly_profile,hourly_profile has its documented row grain,5760,PASS
6,SRC-outreach,outreach has its documented row grain,120,PASS
7,KEY-parking_performance,parking_performance has one row per parking_id,0,PASS
8,KEY-acquisition_scores,acquisition_scores has one row per parking_id,0,PASS
9,KEY-component_scores,component_scores has one row per parking_id,0,PASS


,dataset,row_count,column_count,missing_cells,columns_with_missing,duplicate_full_rows,suspicious_columns
0,acquisition_scores,120,104,18,3,0,36
1,acquisition_terms,120,12,0,0,0,3
2,competition,120,12,132,3,0,6
3,component_scores,120,141,18,3,0,70
4,daily_performance,43800,12,0,0,0,4
5,hourly_profile,5760,8,0,0,0,1
6,locality_summary,17,21,17,1,0,6
7,location_demand,120,15,0,0,0,5
8,outreach,120,27,324,5,0,9
9,owners,72,10,0,0,0,2


In [3]:
metrics = ['avg_occupancy_rate', 'capacity_cars', 'hourly_rate_inr', 'acquisition_score']
display(profile[(profile['dataset'] == 'acquisition_scores') & profile['column'].isin(metrics)][
    ['column', 'missing_count', 'unique_count', 'min', 'p05', 'median', 'mean', 'p95', 'max', 'std', 'skew']
].reset_index(drop=True))
display(suspicious[suspicious['suspicious_flags'].str.contains('HIGH_MISSINGNESS|ZERO_INFLATED', regex=True)][
    ['dataset', 'column', 'missing_pct', 'zero_pct', 'suspicious_flags']
].head(25))

,column,missing_count,unique_count,min,p05,median,mean,p95,max,std,skew
0,capacity_cars,0,104,27.000000,56.700000,149.00000,197.375000,457.700000,706.000000,139.010009,1.216808
1,hourly_rate_inr,0,19,15.000000,25.000000,45.00000,50.666667,85.250000,125.000000,19.893977,1.265864
2,avg_occupancy_rate,0,120,0.113928,0.179046,0.30984,0.338952,0.580281,0.671702,0.122130,0.644952
3,acquisition_score,0,119,20.220000,26.996000,42.52500,45.174417,71.556000,78.530000,14.329112,0.588111


,dataset,column,missing_pct,zero_pct,suspicious_flags
26,acquisition_scores,office_count_500m,0.00,70.833333,ZERO_INFLATED|SKEWED
27,acquisition_scores,rank_stability_pct,0.00,88.333333,ZERO_INFLATED|SKEWED|UPPER_BOUND_CONCENTRATION
30,acquisition_scores,retail_count_500m,0.00,75.833333,ZERO_INFLATED|SKEWED|UPPER_BOUND_CONCENTRATION
33,acquisition_scores,weekday_busy_hour_share,0.00,73.333333,ZERO_INFLATED
34,acquisition_scores,weekend_busy_hour_share,0.00,77.500000,ZERO_INFLATED|SKEWED
42,competition,competitor_total_capacity_1km,100.00,NaN,HIGH_MISSINGNESS|CONSTANT
65,component_scores,demand_headroom_score,0.00,58.333333,ZERO_INFLATED|SKEWED
89,component_scores,office_count_500m,0.00,70.833333,ZERO_INFLATED|SKEWED
98,component_scores,retail_count_500m,0.00,75.833333,ZERO_INFLATED|SKEWED|UPPER_BOUND_CONCENTRATION
109,component_scores,transit_low,0.00,100.000000,CONSTANT|ZERO_INFLATED


## Key findings

- All required source-grain checks pass: 120 lots, 43,800 daily rows, 5,760 hourly rows, and one score row per lot.
- Average lot occupancy ranges from 11.39% to 67.17%; the portfolio median is approximately 29.6%.
- Revenue, bookings, capacity, and OSM POI counts are right-skewed. This is plausible for facility and location data and is explicitly reviewed rather than deleted.
- Competitor capacity is entirely unavailable; competition analysis therefore uses documented count, distance, price, and aggregator proxies.
- Funnel nulls are stage-dependent: conversion fields are empty for leads that have not been won, so those nulls are expected rather than imputed.

## Limitations

Operational, economic, owner, network, and outreach data are synthetic. OSM search coverage is bounded, so sparse or zero POI counts do not prove absence in the real world. Distribution checks identify suspicious shapes; they do not establish that an observation is erroneous.